# 🎙️ MeetScribe — AI Meeting Intelligence Tool
## AlgoProfessor AI Internship | Milestone 4 | Day 25

### 📌 What this notebook does:
**Audio file → Whisper Transcription → Speaker Diarisation → LLM Summary (FREE via Groq) → PDF Report → Gradio UI**

### ✅ Works 100% FREE:
| Tool | Purpose | Cost |
|------|---------|------|
| faster-whisper | Speech-to-text | FREE (runs locally) |
| pyannote.audio | Speaker diarisation | FREE (needs HF token) |
| **Groq API (Llama 3)** | **LLM Summarisation** | **100% FREE** |
| Anthropic Claude | LLM Summarisation | Optional ($5 free credit) |
| OpenAI GPT-4o | LLM Summarisation | Optional (paid) |
| ReportLab | PDF generation | FREE |
| Gradio | Web UI | FREE |

### 🔑 API Keys Needed (get these before running):
1. **Groq (FREE - MAIN LLM)** → https://console.groq.com → API Keys → Create Key
2. **HuggingFace Token** → https://huggingface.co/settings/tokens → New Token (read)
   - Then visit https://huggingface.co/pyannote/speaker-diarization-3.1 → Accept License
3. **Anthropic Claude (Optional)** → https://console.anthropic.com → API Keys ($5 free)
4. **OpenAI (Optional/Skip)** → Only if you have credits (new accounts have none)

---
> ⚠️ **Run cells ONE BY ONE from top to bottom. Do NOT click Run All.**


## 📦 Cell 1 — Install All Required Libraries
> ⏱️ Takes 3–5 minutes. Run once per Colab session. You'll see lots of output — that's normal.

In [ ]:
# Install all required packages
# faster-whisper = 4x faster than original whisper, same accuracy
!pip install -q faster-whisper
!pip install -q pyannote.audio
!pip install -q gradio
!pip install -q reportlab
!pip install -q groq
!pip install -q anthropic
!pip install -q openai
!pip install -q pydub
!pip install -q librosa soundfile
!pip install -q requests

print("=" * 50)
print("✅ ALL PACKAGES INSTALLED SUCCESSFULLY!")
print("=" * 50)


## 🔑 Cell 2 — Import Libraries & Set API Keys

### How to add secrets in Colab:
1. Click the **🔑 key icon** in the left sidebar (Secrets)
2. Add each key with the exact names shown below
3. Toggle the switch ON for each key
4. Then run this cell

> If you don't have a key for a service, just leave it — the notebook handles missing keys gracefully.


In [ ]:
import os
import json
import time
import tempfile
import warnings
import concurrent.futures
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

# ── Audio ──
import librosa
import soundfile as sf
from pydub import AudioSegment

# ── Visualisation ──
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── PDF ──
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import (SimpleDocTemplate, Paragraph,
                                 Spacer, Table, TableStyle, Image as RLImage)
from reportlab.lib import colors

# ── LLM clients ──
from groq import Groq
import anthropic
import openai
import requests

# ────────────────────────────────────────────
# LOAD API KEYS FROM COLAB SECRETS
# ────────────────────────────────────────────
def get_secret(name):
    """Safely get a Colab secret. Returns None if not found."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        return val if val else None
    except Exception:
        return os.environ.get(name)

GROQ_API_KEY      = get_secret('GROQ_API_KEY')       # FREE — REQUIRED
HF_TOKEN          = get_secret('HF_TOKEN')            # Required for diarisation
ANTHROPIC_API_KEY = get_secret('ANTHROPIC_API_KEY')  # Optional
OPENAI_API_KEY    = get_secret('OPENAI_API_KEY')      # Optional

# Initialise clients
groq_client      = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
claude_client    = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None
openai_client    = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

# ── Status Report ──
print("=" * 50)
print("🔑  API KEY STATUS")
print("=" * 50)
print(f"  Groq (Llama 3 FREE)  : {'✅ Ready' if GROQ_API_KEY else '❌ MISSING — Add GROQ_API_KEY to Secrets'}")
print(f"  HuggingFace Token    : {'✅ Ready' if HF_TOKEN else '⚠️  Missing — diarisation will use mock mode'}")
print(f"  Anthropic Claude     : {'✅ Ready' if ANTHROPIC_API_KEY else '⚠️  Optional — skipped'}")
print(f"  OpenAI GPT-4o        : {'✅ Ready' if OPENAI_API_KEY else '⚠️  Optional — skipped (no free tier)'}")
print("=" * 50)
if not GROQ_API_KEY:
    print("\n🚨 GROQ KEY IS REQUIRED!")
    print("   Get your free key at: https://console.groq.com")
    print("   Then add it to Colab Secrets as: GROQ_API_KEY")
else:
    print("\n✅ Ready to proceed! All critical keys loaded.")


## 🎵 Cell 3 — Get a Sample Meeting Audio File

We'll download a **real publicly available audio file** for testing.

> **OR** upload your own: Click the folder icon (📁) on the left → Upload → then update `AUDIO_PATH` below.


In [ ]:
import os

# ─────────────────────────────────────────────
# OPTION A: Download a sample audio (runs automatically)
# ─────────────────────────────────────────────
SAMPLE_URL = "https://www2.cs.uic.edu/~i101/SoundFiles/preamble10.wav"
SAMPLE_PATH = "/content/sample_audio.wav"

print("📥 Downloading sample audio...")
!wget -q -O {SAMPLE_PATH} "{SAMPLE_URL}"

if os.path.exists(SAMPLE_PATH):
    size_kb = os.path.getsize(SAMPLE_PATH) / 1024
    print(f"✅ Sample audio downloaded: {size_kb:.1f} KB")
else:
    print("❌ Download failed. Please upload your own audio file.")

# ─────────────────────────────────────────────
# OPTION B: Use your own uploaded file
# Uncomment and update the path below:
# ─────────────────────────────────────────────
# SAMPLE_PATH = "/content/your_meeting.mp3"   # ← change this

# Final path to use
AUDIO_PATH = SAMPLE_PATH
print(f"\n🎙️ Will process: {AUDIO_PATH}")


## 🔧 Cell 4 — Audio Preprocessing
**Why?** Whisper requires audio at exactly **16kHz sample rate, mono channel**.
If you feed it 44.1kHz stereo (default for most recordings), transcription quality drops badly.
This cell fixes that automatically for any format (mp3, wav, m4a, ogg, flac).


In [ ]:
def preprocess_audio(audio_path: str) -> str:
    """
    Convert any audio file to 16kHz mono WAV.
    Whisper was trained on 16kHz audio — wrong sample rate = bad transcription.

    Returns: path to the processed WAV file
    """
    output_path = "/content/audio_processed_16k.wav"

    print(f"🔧 Preprocessing: {audio_path}")

    # librosa.load handles any format and resamples automatically
    audio, original_sr = librosa.load(audio_path, sr=16000, mono=True)
    # sr=16000  → forces resample to 16kHz
    # mono=True → merges stereo L+R into single channel

    sf.write(output_path, audio, 16000)

    duration_sec = len(audio) / 16000
    duration_min = duration_sec / 60

    print(f"  Original sample rate  : {original_sr} Hz")
    print(f"  Output sample rate    : 16,000 Hz (Whisper-ready)")
    print(f"  Audio duration        : {duration_min:.1f} minutes ({duration_sec:.0f} seconds)")
    print(f"  Channels              : Mono")
    print(f"  Saved to              : {output_path}")
    print(f"✅ Preprocessing complete!")

    return output_path, duration_sec

PROCESSED_AUDIO, AUDIO_DURATION = preprocess_audio(AUDIO_PATH)


## 🤖 Cell 5 — Load Whisper Model
**Why faster-whisper?** It's 4x faster than the original OpenAI Whisper with identical accuracy.
It uses CTranslate2 engine and int8 quantisation to reduce memory usage.

| Model Size | RAM Needed | Speed | Accuracy |
|------------|-----------|-------|----------|
| tiny | 1 GB | Fastest | Good |
| **base** | **1 GB** | **Fast** | **Very Good** ← We use this |
| small | 2 GB | Moderate | Great |
| large-v3 | 6 GB | Slow | Best |

> ⏱️ First run downloads the model (~145MB). Subsequent runs are instant.


In [ ]:
from faster_whisper import WhisperModel

print("⏳ Loading Whisper 'base' model...")
print("   (Downloading ~145MB on first run — please wait)")

# device="cpu"        → works on free Colab (no GPU needed)
# compute_type="int8" → 8-bit quantisation: 2x faster, half memory, same accuracy
whisper_model = WhisperModel(
    "base",
    device="cpu",
    compute_type="int8"
)

print("✅ Whisper model loaded and ready!")
print("   Model: faster-whisper base (int8 quantised)")
print("   Supports: 99+ languages with auto-detection")


## 🎙️ Cell 6 — Transcribe Audio with Whisper
This cell converts your audio to text with **timestamps for every segment**.
We need timestamps to later match each text segment with the correct speaker.

**Key parameter: `word_timestamps=True`** — this is what makes speaker attribution possible.


In [ ]:
def transcribe_audio(audio_path: str) -> dict:
    """
    Transcribe audio using faster-whisper.

    Returns dict:
    {
        'full_text': str,           # Complete transcript
        'segments': [               # List of timed segments
            {'start': 0.0, 'end': 2.5, 'text': 'Hello everyone...'}
        ],
        'language': 'en',           # Auto-detected language
        'duration': 120.5           # Total audio duration in seconds
    }
    """
    print("🎙️ Transcribing audio with Whisper...")
    print("   Please wait — processing audio...")
    start_time = time.time()

    # Run transcription
    segments_gen, info = whisper_model.transcribe(
        audio_path,
        beam_size=5,           # 5 = good accuracy/speed balance (1=fastest, 10=most accurate)
        word_timestamps=True,  # ← CRITICAL: needed to align with speaker segments later
        language=None,         # None = auto-detect language
        vad_filter=True,       # Voice Activity Detection: skip silent parts automatically
        vad_parameters=dict(min_silence_duration_ms=500)  # Skip silences > 0.5 sec
    )

    # Collect segments from generator (must iterate to get results)
    segments = []
    full_text = ""

    for seg in segments_gen:
        segment_data = {
            "start": round(seg.start, 2),
            "end":   round(seg.end,   2),
            "text":  seg.text.strip(),
        }
        segments.append(segment_data)
        full_text += seg.text + " "
        # Show progress
        print(f"   [{seg.start:.1f}s → {seg.end:.1f}s] {seg.text.strip()[:60]}")

    elapsed = time.time() - start_time

    print(f"\n{'='*55}")
    print(f"✅ TRANSCRIPTION COMPLETE")
    print(f"{'='*55}")
    print(f"  Language detected : {info.language} (confidence: {info.language_probability:.1%})")
    print(f"  Total segments    : {len(segments)}")
    print(f"  Audio duration    : {info.duration:.1f} seconds")
    print(f"  Processing time   : {elapsed:.1f} seconds")
    print(f"  Speed ratio       : {info.duration/elapsed:.1f}x realtime")
    print(f"\n📝 Full transcript preview:")
    print("-" * 55)
    print(full_text[:500] + ("..." if len(full_text) > 500 else ""))

    return {
        "full_text": full_text.strip(),
        "segments":  segments,
        "language":  info.language,
        "duration":  info.duration,
    }

TRANSCRIPT = transcribe_audio(PROCESSED_AUDIO)


## 👥 Cell 7 — Speaker Diarisation with pyannote.audio

**What is diarisation?** It answers "who spoke when" — without it, you just have raw text with no speaker labels.

### Prerequisites (do this BEFORE running this cell):
1. Go to → https://huggingface.co/pyannote/speaker-diarization-3.1
2. Click **"Agree and access repository"**
3. Add your `HF_TOKEN` to Colab Secrets

> ⏱️ First run downloads ~1GB model. If HF_TOKEN is missing, we use **mock diarisation** (alternating speakers) so you can still complete the project.


In [ ]:
def run_real_diarisation(audio_path: str, hf_token: str, num_speakers=None) -> list:
    """
    Real speaker diarisation using pyannote.audio 3.1.
    Requires: HF token + accepted pyannote model license on HuggingFace.
    """
    from pyannote.audio import Pipeline

    print("👥 Loading pyannote speaker diarisation pipeline...")
    print("   (Downloads ~1GB model on first run)")

    pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1",
        use_auth_token=hf_token
    )

    print("   Running diarisation on audio...")
    params = {}
    if num_speakers and num_speakers > 1:
        params["num_speakers"] = num_speakers

    diarisation = pipeline(audio_path, **params)

    segments = []
    for turn, _, speaker in diarisation.itertracks(yield_label=True):
        segments.append({
            "start":   round(turn.start, 2),
            "end":     round(turn.end,   2),
            "speaker": speaker,
        })

    speakers_found = set(s["speaker"] for s in segments)
    print(f"✅ Diarisation complete!")
    print(f"   Detected {len(speakers_found)} speakers: {speakers_found}")
    print(f"   Total speaker segments: {len(segments)}")
    return segments


def run_mock_diarisation(transcript_segments: list) -> list:
    """
    Mock diarisation for testing without HF token.
    Alternates between SPEAKER_00 and SPEAKER_01.
    Good enough to test the rest of the pipeline.
    """
    print("⚠️  Using MOCK diarisation (no HF token found)")
    print("   Speakers: alternating SPEAKER_00 / SPEAKER_01")

    speakers = ["SPEAKER_00", "SPEAKER_01", "SPEAKER_02"]
    segments = []
    for i, seg in enumerate(transcript_segments):
        segments.append({
            "start":   seg["start"],
            "end":     seg["end"],
            "speaker": speakers[i % 2],  # Alternate 2 speakers
        })
    print(f"✅ Mock diarisation applied to {len(segments)} segments")
    return segments


# ── Decide which diarisation to use ──
print("=" * 55)
print("SPEAKER DIARISATION")
print("=" * 55)

if HF_TOKEN:
    try:
        SPEAKER_SEGMENTS = run_real_diarisation(
            PROCESSED_AUDIO,
            hf_token=HF_TOKEN,
            num_speakers=None   # ← Change to e.g. 3 if you know speaker count
        )
        DIARISATION_MODE = "real"
    except Exception as e:
        print(f"\n❌ Real diarisation failed: {e}")
        print("   Falling back to mock mode...")
        SPEAKER_SEGMENTS = run_mock_diarisation(TRANSCRIPT["segments"])
        DIARISATION_MODE = "mock"
else:
    SPEAKER_SEGMENTS = run_mock_diarisation(TRANSCRIPT["segments"])
    DIARISATION_MODE = "mock"

print(f"\nDiarisation mode: {DIARISATION_MODE.upper()}")
print("\nFirst 5 speaker segments:")
for seg in SPEAKER_SEGMENTS[:5]:
    print(f"  {seg['speaker']}: [{seg['start']:.1f}s → {seg['end']:.1f}s]")


## 🔀 Cell 8 — Merge Transcript + Speaker Labels

**The key algorithmic step:** Match each Whisper text segment (which has timestamps)
with the pyannote speaker segment that overlaps most with it.

**Logic:** For each text segment, find which speaker was talking at the midpoint of that segment's time window.


In [ ]:
def merge_transcript_and_speakers(
    transcript_segments: list,
    speaker_segments: list
) -> list:
    """
    Combine Whisper text segments with pyannote speaker labels.

    For each transcript segment:
    - Calculate its midpoint timestamp
    - Find which speaker segment contains that midpoint
    - Assign that speaker label to the text

    Then merge consecutive blocks from the same speaker (cleaner output).
    """
    merged = []

    for t_seg in transcript_segments:
        t_mid = (t_seg["start"] + t_seg["end"]) / 2
        assigned_speaker = "UNKNOWN"

        for s_seg in speaker_segments:
            if s_seg["start"] <= t_mid <= s_seg["end"]:
                assigned_speaker = s_seg["speaker"]
                break

        merged.append({
            "speaker": assigned_speaker,
            "start":   t_seg["start"],
            "end":     t_seg["end"],
            "text":    t_seg["text"],
        })

    # ── Merge consecutive same-speaker blocks ──
    # Before: [SPEAKER_00: "Hello"] [SPEAKER_00: "everyone"]
    # After:  [SPEAKER_00: "Hello everyone"]
    combined = []
    for seg in merged:
        if combined and combined[-1]["speaker"] == seg["speaker"]:
            combined[-1]["text"] += " " + seg["text"]
            combined[-1]["end"]   = seg["end"]
        else:
            combined.append(seg.copy())

    print("=" * 55)
    print("DIARISED TRANSCRIPT")
    print("=" * 55)
    print(f"Merged {len(transcript_segments)} segments → {len(combined)} speaker blocks\n")

    for block in combined:
        duration = round(block['end'] - block['start'], 1)
        print(f"[{block['speaker']}] ({block['start']:.1f}s–{block['end']:.1f}s, {duration}s)")
        print(f"  {block['text'][:100]}{'...' if len(block['text'])>100 else ''}")
        print()

    return combined


DIARISED_TRANSCRIPT = merge_transcript_and_speakers(
    TRANSCRIPT["segments"],
    SPEAKER_SEGMENTS
)

# Save intermediate result (so you don't lose it if later cells fail)
with open("/content/diarised_transcript.json", "w") as f:
    json.dump(DIARISED_TRANSCRIPT, f, indent=2)
print("💾 Saved to /content/diarised_transcript.json")


## 📝 Cell 9 — Build the LLM Summarisation Prompt

A well-crafted prompt is the difference between a generic summary and a structured, actionable meeting report.
This prompt instructs the LLM to extract: summary, decisions, action items, sentiment, and topics.

> **You can customise `MEETING_CONTEXT` below** to tell the LLM what the meeting was about.


In [ ]:
# ── Customise this for your meeting ──
MEETING_CONTEXT = "Business / team meeting"
# Examples: "Q3 product planning meeting", "Job interview", "Customer support call"

def build_prompt(diarised_transcript: list, context: str) -> str:
    """Build a structured summarisation prompt from the diarised transcript."""

    # Format transcript as "SPEAKER_XX: text" lines
    transcript_text = "\n".join([
        f"[{seg['speaker']}] ({seg['start']:.0f}s): {seg['text']}"
        for seg in diarised_transcript
    ])

    prompt = f"""You are an expert meeting analyst. Carefully read the transcript below and produce a structured meeting report.

## MEETING CONTEXT
{context}

## YOUR TASK
Provide the following sections clearly:

### 1. EXECUTIVE SUMMARY
Write 3-4 sentences summarising the entire meeting.

### 2. KEY DECISIONS MADE
List any decisions agreed upon. Use bullet points.
If none: write "No formal decisions identified."

### 3. ACTION ITEMS
Format each as: "- [SPEAKER_XX] must [action] by [deadline if mentioned]"
If none: write "No explicit action items identified."

### 4. TOPICS DISCUSSED
Brief bullet list of all topics covered.

### 5. MEETING SENTIMENT
Overall tone (Positive / Neutral / Negative / Mixed) and a 1-sentence explanation.

---
## TRANSCRIPT
{transcript_text}
---

Respond in the exact format above. Be concise, professional, and actionable."""

    return prompt

PROMPT = build_prompt(DIARISED_TRANSCRIPT, MEETING_CONTEXT)

print("✅ Prompt built successfully!")
print(f"   Prompt length: {len(PROMPT)} characters")
print(f"   Meeting context: {MEETING_CONTEXT}")
print("\n--- PROMPT PREVIEW (first 400 chars) ---")
print(PROMPT[:400] + "...")


## 🤖 Cell 10 — 3-Way LLM Summarisation (FREE + Optional Paid)

This calls all available LLMs with the **same prompt** so you can compare quality, speed, and cost.

| Model | Provider | Cost |
|-------|---------|------|
| **Llama 3.3 70B** | **Groq** | **FREE ✅** |
| Claude 3.5 Sonnet | Anthropic | ~$0.001 |
| GPT-4o | OpenAI | ~$0.002 |

> Models without API keys are automatically skipped — no errors.


In [ ]:
def call_groq_llama3(prompt: str) -> dict:
    """FREE: Llama 3.3 70B via Groq API. 400+ tokens/sec."""
    if not groq_client:
        return {"model": "Llama 3.3-70B (Groq FREE)", "status": "skipped — no GROQ_API_KEY"}

    start = time.time()
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1000,
            temperature=0.3,
        )
        return {
            "model":       "Llama 3.3-70B (Groq FREE)",
            "summary":     response.choices[0].message.content,
            "tokens":      response.usage.total_tokens,
            "cost_usd":    0.0,
            "latency_sec": round(time.time() - start, 2),
            "status":      "success",
        }
    except Exception as e:
        return {"model": "Llama 3.3-70B (Groq FREE)", "status": f"error: {e}"}


def call_claude(prompt: str) -> dict:
    """Anthropic Claude 3.5 Sonnet (~$5 free credit on new accounts)."""
    if not claude_client:
        return {"model": "Claude 3.5 Sonnet", "status": "skipped — no ANTHROPIC_API_KEY"}

    start = time.time()
    try:
        response = claude_client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=1000,
            messages=[{"role": "user", "content": prompt}],
        )
        in_tok  = response.usage.input_tokens
        out_tok = response.usage.output_tokens
        cost    = round(in_tok * 0.000003 + out_tok * 0.000015, 5)
        return {
            "model":       "Claude 3.5 Sonnet",
            "summary":     response.content[0].text,
            "tokens":      in_tok + out_tok,
            "cost_usd":    cost,
            "latency_sec": round(time.time() - start, 2),
            "status":      "success",
        }
    except Exception as e:
        return {"model": "Claude 3.5 Sonnet", "status": f"error: {e}"}


def call_gpt4o(prompt: str) -> dict:
    """OpenAI GPT-4o (only if you have credits — optional)."""
    if not openai_client:
        return {"model": "GPT-4o", "status": "skipped — no OPENAI_API_KEY"}

    start = time.time()
    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1000,
            temperature=0.3,
        )
        tok  = response.usage.total_tokens
        cost = round(tok * 0.000005, 5)
        return {
            "model":       "GPT-4o",
            "summary":     response.choices[0].message.content,
            "tokens":      tok,
            "cost_usd":    cost,
            "latency_sec": round(time.time() - start, 2),
            "status":      "success",
        }
    except Exception as e:
        return {"model": "GPT-4o", "status": f"error: {e}"}


# ── Call all LLMs in parallel ──
print("🤖 Calling all available LLMs simultaneously...")
print("   (Parallel execution — won't take 3x longer)\n")

with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    futures = [
        executor.submit(call_groq_llama3, PROMPT),
        executor.submit(call_claude,      PROMPT),
        executor.submit(call_gpt4o,       PROMPT),
    ]
    LLM_RESULTS = {}
    for future in concurrent.futures.as_completed(futures):
        result = future.result()
        LLM_RESULTS[result["model"]] = result
        status = "✅" if result["status"] == "success" else ("⚠️ " if "skip" in result["status"] else "❌")
        lat  = result.get("latency_sec", "—")
        cost = result.get("cost_usd", "—")
        print(f"  {status} {result['model']:35s} | {lat}s | ${cost}")

print("\n" + "="*55)
print("LLM SUMMARIES")
print("="*55)
for model, result in LLM_RESULTS.items():
    if result["status"] == "success":
        print(f"\n{'─'*55}")
        print(f"📌 {result['model']}")
        print(f"{'─'*55}")
        print(result["summary"][:600] + ("..." if len(result["summary"]) > 600 else ""))

# Save results
with open("/content/llm_results.json", "w") as f:
    json.dump(LLM_RESULTS, f, indent=2, default=str)
print("\n💾 All LLM results saved to /content/llm_results.json")


## 📊 Cell 11 — Meeting Analytics Visualisation

Generates 3 charts:
1. **Speaker Talk-Time** — Pie chart of how much each speaker talked
2. **Speaking Timeline** — Gantt-style chart showing who spoke when
3. **LLM Comparison** — Latency and cost comparison bar chart


In [ ]:
import os
os.makedirs("/content/outputs", exist_ok=True)

def generate_analytics_charts(diarised_transcript, llm_results, save_path="/content/outputs/"):
    """Generate meeting analytics visualisation dashboard."""

    COLORS = ['#1F3A8C', '#E85D04', '#2E7D32', '#6A1B9A', '#00695C',
              '#C62828', '#F57F17', '#0277BD']

    fig = plt.figure(figsize=(18, 6))
    fig.suptitle("MeetScribe — Meeting Analytics Dashboard",
                 fontsize=15, fontweight='bold', y=1.02)

    # ── Plot 1: Speaker Talk Time (Pie) ──────────────────────
    ax1 = fig.add_subplot(1, 3, 1)
    speaker_time = {}
    for seg in diarised_transcript:
        spk = seg["speaker"]
        dur = round(seg["end"] - seg["start"], 2)
        speaker_time[spk] = speaker_time.get(spk, 0) + dur

    speakers = list(speaker_time.keys())
    times    = list(speaker_time.values())
    clrs     = COLORS[:len(speakers)]

    wedges, texts, autotexts = ax1.pie(
        times, labels=speakers, colors=clrs,
        autopct='%1.1f%%', startangle=90,
        textprops={'fontsize': 9}
    )
    ax1.set_title("Speaker Talk-Time Distribution", fontweight='bold', fontsize=11)

    # Legend with seconds
    legend_labels = [f"{spk}: {t:.0f}s" for spk, t in speaker_time.items()]
    ax1.legend(legend_labels, loc='lower center',
               bbox_to_anchor=(0.5, -0.15), fontsize=8, ncol=1)

    # ── Plot 2: Speaking Timeline (Gantt) ─────────────────────
    ax2 = fig.add_subplot(1, 3, 2)
    color_map = {spk: COLORS[i % len(COLORS)] for i, spk in enumerate(speakers)}

    for seg in diarised_transcript:
        width = max(seg["end"] - seg["start"], 0.1)
        ax2.barh(
            seg["speaker"], width,
            left=seg["start"],
            color=color_map[seg["speaker"]],
            alpha=0.75, height=0.5, edgecolor='white'
        )

    ax2.set_xlabel("Time (seconds)", fontsize=9)
    ax2.set_title("Speaking Timeline", fontweight='bold', fontsize=11)
    ax2.grid(axis='x', alpha=0.3, linestyle='--')
    ax2.invert_yaxis()

    # ── Plot 3: LLM Latency Comparison ────────────────────────
    ax3 = fig.add_subplot(1, 3, 3)
    successful = {k: v for k, v in llm_results.items() if v.get("status") == "success"}

    if successful:
        model_names = [v["model"].split("(")[0].strip()[:15] for v in successful.values()]
        latencies   = [v["latency_sec"] for v in successful.values()]
        costs       = [v["cost_usd"] for v in successful.values()]
        clrs2       = COLORS[:len(model_names)]

        bars = ax3.bar(range(len(model_names)), latencies, color=clrs2, alpha=0.85, edgecolor='white')
        ax3.set_xticks(range(len(model_names)))
        ax3.set_xticklabels(model_names, rotation=15, ha='right', fontsize=8)
        ax3.set_ylabel("Latency (seconds)", fontsize=9)
        ax3.set_title("LLM Speed & Cost Comparison", fontweight='bold', fontsize=11)
        ax3.grid(axis='y', alpha=0.3, linestyle='--')

        for bar, cost in zip(bars, costs):
            label = "FREE" if cost == 0 else f"${cost:.5f}"
            ax3.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.05,
                label, ha='center', va='bottom',
                fontsize=8, fontweight='bold',
                color='green' if cost == 0 else '#333333'
            )
    else:
        ax3.text(0.5, 0.5, "No successful\nLLM calls",
                 ha='center', va='center', transform=ax3.transAxes, fontsize=12)

    plt.tight_layout()
    chart_path = f"{save_path}meeting_analytics.png"
    plt.savefig(chart_path, dpi=150, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.show()
    print(f"\n✅ Analytics chart saved to: {chart_path}")
    return chart_path

CHART_PATH = generate_analytics_charts(DIARISED_TRANSCRIPT, LLM_RESULTS)


## 📄 Cell 12 — Generate PDF Meeting Report

Creates a professional, formatted PDF with:
- Meeting metadata and analytics chart
- Speaker-attributed transcript
- All LLM summaries side by side
- Performance comparison table


In [ ]:
def generate_pdf_report(
    diarised_transcript: list,
    llm_results: dict,
    transcript_info: dict,
    chart_path: str,
    output_path: str = "/content/outputs/meeting_report.pdf"
) -> str:
    """Generate a professional PDF meeting minutes report."""

    doc = SimpleDocTemplate(
        output_path, pagesize=letter,
        rightMargin=0.75*inch, leftMargin=0.75*inch,
        topMargin=0.75*inch,   bottomMargin=0.75*inch
    )
    styles = getSampleStyleSheet()

    # Custom styles
    title_style = ParagraphStyle('MTitle', parent=styles['Title'],
        fontSize=20, textColor=colors.HexColor('#1F3A8C'), spaceAfter=10)
    meta_style  = ParagraphStyle('MMeta', parent=styles['Normal'],
        fontSize=9, textColor=colors.grey, spaceAfter=16)
    h2_style    = ParagraphStyle('MH2', parent=styles['Heading2'],
        fontSize=13, textColor=colors.HexColor('#E85D04'), spaceAfter=8, spaceBefore=14)
    body_style  = ParagraphStyle('MBody', parent=styles['Normal'],
        fontSize=9.5, spaceAfter=5, leading=14)
    spk_style   = ParagraphStyle('MSpk', parent=styles['Normal'],
        fontSize=9.5, spaceAfter=4, leading=13,
        textColor=colors.HexColor('#333333'))

    story = []
    now   = datetime.now().strftime("%B %d, %Y  %H:%M")
    dur_m = transcript_info.get("duration", 0) / 60
    lang  = transcript_info.get("language", "en").upper()

    story.append(Paragraph("📋  MeetScribe — AI Meeting Intelligence Report", title_style))
    story.append(Paragraph(
        f"Generated: {now}  |  Duration: {dur_m:.1f} min  |  "
        f"Language: {lang}  |  Speakers detected: "
        f"{len(set(s['speaker'] for s in diarised_transcript))}",
        meta_style
    ))

    # ── Chart ──
    story.append(Paragraph("Meeting Analytics Dashboard", h2_style))
    story.append(RLImage(chart_path, width=6.5*inch, height=2.1*inch))
    story.append(Spacer(1, 0.15*inch))

    # ── Attributed Transcript ──
    story.append(Paragraph("Diarised Transcript (Speaker-Attributed)", h2_style))
    max_blocks = min(len(diarised_transcript), 15)  # First 15 blocks in PDF
    for seg in diarised_transcript[:max_blocks]:
        ts = f"({seg['start']:.0f}s–{seg['end']:.0f}s)"
        txt = seg['text'].replace('<', '&lt;').replace('>', '&gt;')
        story.append(Paragraph(
            f"<b>[{seg['speaker']}]</b> <font size=8 color=grey>{ts}</font>:  {txt}",
            spk_style
        ))
    if len(diarised_transcript) > max_blocks:
        story.append(Paragraph(
            f"<i>... {len(diarised_transcript)-max_blocks} more blocks. "
            f"See /content/diarised_transcript.json for full transcript.</i>",
            meta_style
        ))
    story.append(Spacer(1, 0.15*inch))

    # ── LLM Summaries ──
    story.append(Paragraph("LLM Summarisation Comparison", h2_style))
    for model_name, result in llm_results.items():
        if result.get("status") == "success":
            cost_label = "FREE" if result["cost_usd"] == 0 else f"${result['cost_usd']:.5f}"
            story.append(Paragraph(
                f"<b>🤖 {result['model']}</b>  |  "
                f"⏱ {result['latency_sec']}s  |  💰 {cost_label}  |  "
                f"🔤 {result['tokens']} tokens",
                body_style
            ))
            summary_safe = result["summary"].replace('<','&lt;').replace('>','&gt;')
            story.append(Paragraph(summary_safe[:800], body_style))
            story.append(Spacer(1, 0.1*inch))

    # ── Performance Table ──
    story.append(Paragraph("Model Performance Summary", h2_style))
    table_data = [["Model", "Latency", "Cost", "Tokens", "Status"]]
    for result in llm_results.values():
        cost_str = "FREE" if result.get("cost_usd", 0) == 0 else f"${result.get('cost_usd','—'):.5f}"
        table_data.append([
            result["model"][:28],
            f"{result.get('latency_sec','—')}s",
            cost_str,
            str(result.get("tokens", "—")),
            result["status"][:20],
        ])

    t = Table(table_data, colWidths=[2.2*inch, 0.9*inch, 1.0*inch, 0.8*inch, 1.6*inch])
    t.setStyle(TableStyle([
        ('BACKGROUND',    (0,0), (-1,0),  colors.HexColor('#1F3A8C')),
        ('TEXTCOLOR',     (0,0), (-1,0),  colors.white),
        ('FONTNAME',      (0,0), (-1,0),  'Helvetica-Bold'),
        ('FONTSIZE',      (0,0), (-1,-1), 8),
        ('GRID',          (0,0), (-1,-1), 0.4, colors.grey),
        ('ROWBACKGROUNDS',(0,1), (-1,-1), [colors.white, colors.HexColor('#EEF2FF')]),
        ('ALIGN',         (1,0), (-1,-1), 'CENTER'),
        ('VALIGN',        (0,0), (-1,-1), 'MIDDLE'),
        ('TOPPADDING',    (0,0), (-1,-1), 5),
        ('BOTTOMPADDING', (0,0), (-1,-1), 5),
    ]))
    story.append(t)

    doc.build(story)
    print(f"✅ PDF report generated!")
    print(f"   Saved to: {output_path}")
    return output_path

PDF_PATH = generate_pdf_report(
    DIARISED_TRANSCRIPT,
    LLM_RESULTS,
    TRANSCRIPT,
    CHART_PATH
)


## 💬 Cell 13 — Slack Notification (Optional)

Sends a summary to a Slack channel when the meeting is processed.

**Setup (takes 2 minutes):**
1. Go to https://api.slack.com/apps → Create New App → From Scratch
2. Incoming Webhooks → Activate → Add New Webhook → Copy URL
3. Add to Colab Secrets as `SLACK_WEBHOOK_URL`

> If no webhook is configured, this cell skips gracefully.


In [ ]:
def send_slack_notification(llm_results: dict, transcript_info: dict) -> bool:
    """Send meeting summary to Slack. Skips if no webhook configured."""

    SLACK_WEBHOOK = get_secret('SLACK_WEBHOOK_URL')

    if not SLACK_WEBHOOK:
        print("⚠️  SLACK_WEBHOOK_URL not found in Secrets.")
        print("   Skipping Slack notification (this is optional).")
        print("   To enable: add SLACK_WEBHOOK_URL to Colab Secrets.")
        return False

    # Pick best available summary
    best_summary = "Summary unavailable."
    best_model   = "—"
    for model_name in ["Llama 3.3-70B (Groq FREE)", "Claude 3.5 Sonnet", "GPT-4o"]:
        r = llm_results.get(model_name, {})
        if r.get("status") == "success":
            best_summary = r["summary"][:400]
            best_model   = r["model"]
            break

    dur_min = transcript_info.get("duration", 0) / 60

    payload = {
        "blocks": [
            {"type": "header",
             "text": {"type": "plain_text", "text": "🎙️ MeetScribe — New Meeting Processed"}},
            {"type": "section",
             "fields": [
                 {"type": "mrkdwn", "text": f"*Duration:* {dur_min:.1f} minutes"},
                 {"type": "mrkdwn", "text": f"*Language:* {transcript_info.get('language','en').upper()}"},
                 {"type": "mrkdwn", "text": f"*Best Model:* {best_model}"},
                 {"type": "mrkdwn", "text": f"*Processed:* {datetime.now().strftime('%H:%M')}"},
             ]},
            {"type": "section",
             "text": {"type": "mrkdwn",
                      "text": f"*Summary (by {best_model}):*\n{best_summary}..."}},
            {"type": "divider"},
        ]
    }

    response = requests.post(SLACK_WEBHOOK, json=payload, timeout=10)
    if response.status_code == 200:
        print("✅ Slack notification sent successfully!")
        return True
    else:
        print(f"❌ Slack failed: HTTP {response.status_code} — {response.text}")
        return False

send_slack_notification(LLM_RESULTS, TRANSCRIPT)


## 🌐 Cell 14 — Gradio Web Interface (Full MeetScribe App)

This wraps the **entire pipeline** into a clean web UI.

After running this cell:
1. Look for a line like: `Running on public URL: https://xxxx.gradio.live`
2. Click that URL — it works on any device, anywhere in the world
3. Upload an audio file and click "Process Meeting"

> ⏱️ Processing a 5-minute meeting takes about 2–3 minutes.


In [ ]:
import gradio as gr

def meetscribe_pipeline(audio_file, meeting_context, num_speakers_input):
    """
    Full end-to-end pipeline for the Gradio UI.
    Called when user clicks 'Process Meeting'.
    """
    if audio_file is None:
        return ("❌ No audio uploaded.", "", "", None)

    logs = []

    def log(msg):
        logs.append(msg)
        print(msg)

    try:
        # ── Stage 1: Preprocess ────────────────────────────
        log("📥 [1/5] Preprocessing audio to 16kHz mono WAV...")
        processed, duration_sec = preprocess_audio(audio_file)
        log(f"   ✅ Duration: {duration_sec:.1f}s")

        # ── Stage 2: Transcribe ────────────────────────────
        log("🎙️ [2/5] Transcribing with Whisper (please wait)...")
        transcript = transcribe_audio(processed)
        log(f"   ✅ {len(transcript['segments'])} segments, language: {transcript['language']}")

        # ── Stage 3: Diarise ───────────────────────────────
        log("👥 [3/5] Running speaker diarisation...")
        try:
            n = int(num_speakers_input) if num_speakers_input and int(num_speakers_input) > 1 else None
            if HF_TOKEN:
                speakers = run_real_diarisation(processed, HF_TOKEN, num_speakers=n)
            else:
                raise ValueError("No HF token")
        except Exception as e:
            log(f"   ⚠️  Using mock diarisation ({e})")
            speakers = run_mock_diarisation(transcript["segments"])

        # ── Stage 4: Merge + Summarise ─────────────────────
        log("🔀 [4/5] Merging transcript + speakers...")
        diarised = merge_transcript_and_speakers(transcript["segments"], speakers)

        log("🤖 [4/5] Calling LLMs in parallel...")
        ctx    = meeting_context.strip() or "Business meeting"
        prompt = build_prompt(diarised, ctx)

        with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
            futs = [executor.submit(call_groq_llama3, prompt),
                    executor.submit(call_claude, prompt),
                    executor.submit(call_gpt4o, prompt)]
            results = {r.result()["model"]: r.result()
                       for r in concurrent.futures.as_completed(futs)}

        for m, r in results.items():
            status = "✅" if r["status"] == "success" else "⚠️ " + r["status"][:30]
            log(f"   {status} — {m}")

        # ── Stage 5: Export ────────────────────────────────
        log("📊 [5/5] Generating charts + PDF...")
        chart = generate_analytics_charts(diarised, results)
        pdf   = generate_pdf_report(diarised, results, transcript, chart)

        # Format outputs for Gradio
        transcript_out = "\n".join([
            f"[{s['speaker']}] ({s['start']:.0f}s): {s['text']}"
            for s in diarised
        ])

        summary_out = ""
        for m, r in results.items():
            if r.get("status") == "success":
                cost = "FREE" if r["cost_usd"] == 0 else f"${r['cost_usd']:.5f}"
                summary_out += f"{'='*50}\n"
                summary_out += f"🤖 {r['model']}  |  ⏱ {r['latency_sec']}s  |  💰 {cost}\n"
                summary_out += f"{'='*50}\n"
                summary_out += r["summary"] + "\n\n"

        log("✅ [Done] MeetScribe processing complete!")
        status_out = "\n".join(logs)

        return (status_out, transcript_out, summary_out, pdf)

    except Exception as e:
        import traceback
        err = f"❌ Error: {e}\n\n{traceback.format_exc()}"
        return (err, "", "", None)


# ── Build Gradio Interface ──────────────────────────────────
with gr.Blocks(
    title="MeetScribe — AI Meeting Intelligence",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown("""
    # 🎙️ MeetScribe — AI Meeting Intelligence Tool
    ### AlgoProfessor AI Internship | Milestone 4 | Day 25

    **Pipeline:** Upload Audio → Whisper Transcription → Speaker Diarisation
    → 3-Way LLM Summary (Groq FREE + Claude + GPT-4o) → PDF Report

    > ✅ Works 100% FREE with just a Groq API key!
    """)

    with gr.Row():
        # ── Left column: Inputs ──
        with gr.Column(scale=1):
            audio_in = gr.Audio(
                type="filepath",
                label="📁 Upload Meeting Audio (MP3 / WAV / M4A)"
            )
            context_in = gr.Textbox(
                label="Meeting Context (optional)",
                placeholder="e.g. Q3 planning meeting, product team",
                lines=2
            )
            speakers_in = gr.Number(
                label="Expected speakers (0 = auto-detect)",
                value=0, precision=0
            )
            submit_btn = gr.Button(
                "🚀 Process Meeting",
                variant="primary", size="lg"
            )
            gr.Markdown("""
            **⏱️ Expected time:**
            - 5-min audio → ~2 min
            - 30-min audio → ~8 min
            """)

        # ── Right column: Outputs ──
        with gr.Column(scale=2):
            status_out = gr.Textbox(
                label="⚙️ Processing Status",
                lines=8, interactive=False
            )
            transcript_out = gr.Textbox(
                label="📝 Diarised Transcript (Speaker-Attributed)",
                lines=12, interactive=False
            )
            summary_out = gr.Textbox(
                label="🤖 3-Way LLM Summary Comparison",
                lines=15, interactive=False
            )
            pdf_out = gr.File(
                label="📥 Download PDF Meeting Report"
            )

    submit_btn.click(
        fn=meetscribe_pipeline,
        inputs=[audio_in, context_in, speakers_in],
        outputs=[status_out, transcript_out, summary_out, pdf_out]
    )

    gr.Markdown("""
    ---
    **🔑 API Keys needed in Colab Secrets:**
    `GROQ_API_KEY` (FREE at groq.com) · `HF_TOKEN` (huggingface.co) · 
    `ANTHROPIC_API_KEY` (optional) · `OPENAI_API_KEY` (optional)
    """)

# Launch with public sharing
demo.launch(share=True, debug=False, quiet=True)
print("\n🌐 MeetScribe is LIVE!")
print("   Look for the line: 'Running on public URL: https://xxxx.gradio.live'")
print("   Click that URL to open the app!")


## 📥 Cell 15 — Download All Output Files

Run this cell to download all generated files directly to your computer.


In [ ]:
from google.colab import files
import os

print("📦 Downloading all MeetScribe outputs...\n")

files_to_download = {
    "/content/outputs/meeting_report.pdf":    "📄 PDF Meeting Report",
    "/content/outputs/meeting_analytics.png": "📊 Analytics Chart",
    "/content/diarised_transcript.json":      "📝 Diarised Transcript (JSON)",
    "/content/llm_results.json":              "🤖 LLM Comparison Results (JSON)",
}

for filepath, label in files_to_download.items():
    if os.path.exists(filepath):
        print(f"  Downloading: {label}")
        files.download(filepath)
    else:
        print(f"  ⚠️  Not found (may not have been generated): {filepath}")

print("\n✅ All available files sent for download!")
print("   Check your browser's Downloads folder.")
